# Road Accident Pattern Analysis, India (DA-022)
**Domain:** Road Safety Analytics

**Problem:** A road-safety team wants to understand when and where recorded accidents in India are more frequent.

**Objectives**
1. Summarize accident counts and severity
2. Compare time periods (hour, weekday, month) and conditions (weather, road features, cause)
3. Identify data-quality gaps

**Dataset:** Indian Road Accident Dataset (2022-2025), Kaggle (Mansehaj Preet, `sehaj1104`). 20,000 records, 24 columns. No personal information.

**Note:** Descriptive analysis only. Correlation does not prove causation, and no blame is assigned. Charts marked *(Play)* are animated: press the play button under the chart.

## Step 1: Import libraries

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import glob, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## Step 2: Load the dataset from Kaggle
The notebook downloads the dataset directly from Kaggle (`sehaj1104/indian-road-accident-dataset-20222025`).
If the download fails, upload the CSV manually into the `/content` folder (left sidebar, folder icon, upload button) and run this cell again.

In [26]:
import subprocess, os
subprocess.run(['pip', '-q', 'install', 'kagglehub'], capture_output=True)

try:
    import kagglehub
    folder = kagglehub.dataset_download('sehaj1104/indian-road-accident-dataset-20222025')
    csvs = glob.glob(os.path.join(folder, '*.csv'))
    print('Downloaded from Kaggle:', csvs)
except Exception as e:
    print('Kaggle download failed, using the uploaded file instead. Reason:', e)
    csvs = glob.glob('/content/*.csv')

PATH = csvs[0]
df = pd.read_csv(PATH)
print('Shape:', df.shape)
df.head()

Using Colab cache for faster access to the 'indian-road-accident-dataset-20222025' dataset.
Downloaded from Kaggle: ['/kaggle/input/indian-road-accident-dataset-20222025/indian_roads_dataset.csv']
Shape: (20000, 24)


,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,...,visibility,temperature,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,festival,risk_score
0,0,Pune,Maharashtra,18.680827,73.930388,2023-10-22,5:00,5,Sunday,1,...,low,32,high,weather,fatal,2,2,0,NaN,0.85
1,1,Mumbai,Maharashtra,18.817732,72.790846,2023-05-21,4:00,4,Sunday,1,...,high,34,low,weather,major,4,3,0,NaN,0.10
2,2,Mumbai,Maharashtra,19.096889,72.819424,2024-07-10,13:00,13,Wednesday,0,...,low,21,medium,weather,minor,1,1,0,NaN,0.45
3,3,Chandigarh,Punjab,30.787805,76.847507,2025-03-30,11:00,11,Sunday,1,...,low,30,high,distraction,minor,5,2,0,NaN,0.65
4,4,Chennai,Tamil Nadu,12.965155,80.283313,2024-01-25,16:00,16,Thursday,0,...,high,24,low,distraction,minor,2,1,0,NaN,0.10


## Step 3: Inspect the data

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   accident_id        20000 non-null  int64  
 1   city               20000 non-null  object 
 2   state              20000 non-null  object 
 3   latitude           20000 non-null  float64
 4   longitude          20000 non-null  float64
 5   date               20000 non-null  object 
 6   time               20000 non-null  object 
 7   hour               20000 non-null  int64  
 8   day_of_week        20000 non-null  object 
 9   is_weekend         20000 non-null  int64  
 10  road_type          20000 non-null  object 
 11  lanes              20000 non-null  int64  
 12  traffic_signal     20000 non-null  int64  
 13  weather            20000 non-null  object 
 14  visibility         20000 non-null  object 
 15  temperature        20000 non-null  int64  
 16  traffic_density    200

In [28]:
miss = (df.isnull().mean() * 100).round(2)
print('Missing values (%):')
print(miss[miss > 0])
print()
print('Duplicate ids :', df['accident_id'].duplicated().sum())
print('Duplicate rows:', df.duplicated().sum())
print()
print(df['accident_severity'].value_counts())

Missing values (%):
festival    99.42
dtype: float64

Duplicate ids : 0
Duplicate rows: 0

accident_severity
minor    11025
major     5988
fatal     2987
Name: count, dtype: int64


## Step 4: Clean the data

In [29]:
n_raw = len(df)
dup_ids = int(df['accident_id'].duplicated().sum())
df = df.drop_duplicates(subset='accident_id').copy()

# dates and hours
df['date'] = pd.to_datetime(df['date'], errors='coerce')
bad_date = int(df['date'].isnull().sum())
df['hour'] = pd.to_numeric(df['hour'], errors='coerce')
bad_hour = int((~df['hour'].between(0, 23)).sum())

# location check (India lies roughly between lat 6-38 and lon 68-98)
df['geo_ok'] = df['latitude'].between(6, 38) & df['longitude'].between(68, 98)
bad_geo = int((~df['geo_ok']).sum())

df = df.dropna(subset=['date'])
df = df[df['hour'].between(0, 23)].copy()
df['hour'] = df['hour'].astype(int)

# text columns: fill missing and tidy
for c in ['city', 'state', 'weather', 'road_type', 'cause', 'traffic_density', 'visibility', 'traffic_signal']:
    df[c] = df[c].fillna('Unknown').astype(str).str.strip()
df['accident_severity'] = df['accident_severity'].fillna('Unknown').astype(str).str.strip().str.lower()
df['festival'] = df['festival'].fillna('No festival / not recorded')

print('Rows before:', n_raw, '| after cleaning:', len(df))
print('Duplicate ids removed:', dup_ids, '| bad dates:', bad_date, '| bad hours:', bad_hour, '| locations outside India:', bad_geo)

Rows before: 20000 | after cleaning: 20000
Duplicate ids removed: 0 | bad dates: 0 | bad hours: 0 | locations outside India: 0


## Step 5: Feature engineering

In [30]:
df['Weekday'] = df['date'].dt.day_name()
df['Month'] = df['date'].dt.month_name()
df['Year'] = df['date'].dt.year.astype(str)
df['Period'] = pd.cut(df['hour'], [-1, 5, 11, 16, 20, 23],
                      labels=['Night (0-5)', 'Morning (6-11)', 'Afternoon (12-16)', 'Evening (17-20)', 'Late night (21-23)'])

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
month_order = ['January','February','March','April','May','June','July','August','September','October','November','December']

pref = ['minor', 'moderate', 'major', 'fatal']
present = list(df['accident_severity'].unique())
sev_order = [s for s in pref if s in present] + [s for s in present if s not in pref]
severe_set = [s for s in ['major', 'fatal'] if s in present]
df['Severe'] = df['accident_severity'].isin(severe_set)

def order(d, col, cats):
    d[col] = pd.Categorical(d[col], categories=cats, ordered=True)
    return d.sort_values(col)

df[['date', 'hour', 'Weekday', 'Month', 'Period', 'accident_severity', 'Severe']].head()

,date,hour,Weekday,Month,Period,accident_severity,Severe
0,2023-10-22,5,Sunday,October,Night (0-5),fatal,True
1,2023-05-21,4,Sunday,May,Night (0-5),major,True
2,2024-07-10,13,Wednesday,July,Afternoon (12-16),minor,False
3,2025-03-30,11,Sunday,March,Morning (6-11),minor,False
4,2024-01-25,16,Thursday,January,Afternoon (12-16),minor,False


## Step 6: Summary tables and KPIs

In [31]:
sev = df['accident_severity'].value_counts().reindex(sev_order)
hourly = df['hour'].value_counts().reindex(range(24), fill_value=0).sort_index()
wd = df['Weekday'].value_counts().reindex(day_order)
mo = df['Month'].value_counts().reindex(month_order).dropna()
wea = df['weather'].value_counts()
cause = df['cause'].value_counts()
states = df['state'].value_counts().head(10)
cities = df['city'].value_counts().head(10)
period = df['Period'].value_counts()
pivot = df.pivot_table(index='Weekday', columns='hour', values='accident_id', aggfunc='count').reindex(day_order)

sev_wea = (df.groupby('weather')['Severe'].mean() * 100).sort_values(ascending=False)
feat_parts = {}
for col in ['road_type', 'traffic_signal', 'traffic_density']:
    for k, v in (df.groupby(col)['Severe'].mean() * 100).items():
        feat_parts[f'{col}: {k}'] = v
sev_feat = pd.Series(feat_parts).sort_values(ascending=False)

severe_pct = round(float(df['Severe'].mean() * 100), 1)
peak_hour = int(hourly.idxmax())
total_cas = int(df['casualties'].sum())

print('Total accidents :', len(df))
print('Date range      :', df['date'].min().date(), 'to', df['date'].max().date())
print('States / cities :', df['state'].nunique(), '/', df['city'].nunique())
print('Peak hour       :', peak_hour, ':00')
print('Severe (major+fatal) %:', severe_pct)
print('Total casualties:', total_cas)
print()
print((df['accident_severity'].value_counts(normalize=True) * 100).round(1))

Total accidents : 20000
Date range      : 2022-01-01 to 2025-04-15
States / cities : 7 / 8
Peak hour       : 2 :00
Severe (major+fatal) %: 44.9
Total casualties: 34529

accident_severity
minor    55.1
major    29.9
fatal    14.9
Name: proportion, dtype: float64


## Step 7: Charts

### Chart 1: Accidents by severity, year wise *(Play)*

In [32]:
d = df.groupby(['Year', 'accident_severity']).size().reset_index(name='Accidents')
px.bar(d, x='accident_severity', y='Accidents', color='accident_severity', animation_frame='Year',
       category_orders={'accident_severity': sev_order}, range_y=[0, d['Accidents'].max() * 1.1],
       title='Chart 1: Accidents by Severity').show()

### Chart 2: Accidents by hour of day, weekday wise *(Play)*

In [33]:
d = df.groupby(['Weekday', 'hour']).size().reset_index(name='Accidents')
d = order(d, 'Weekday', day_order)
px.line(d, x='hour', y='Accidents', markers=True, animation_frame='Weekday',
        range_y=[0, d['Accidents'].max() * 1.1], title='Chart 2: Accidents by Hour of Day').show()

### Chart 3: Accidents by weekday, year wise *(Play)*

In [34]:
d = df.groupby(['Year', 'Weekday']).size().reset_index(name='Accidents')
px.bar(d, x='Weekday', y='Accidents', animation_frame='Year', category_orders={'Weekday': day_order},
       range_y=[0, d['Accidents'].max() * 1.1], title='Chart 3: Accidents by Day of Week').show()

### Chart 4: Heatmap, weekday vs hour (hover for exact counts)

In [35]:
px.imshow(pivot, color_continuous_scale='YlOrRd', aspect='auto',
          labels=dict(x='Hour', y='Weekday', color='Accidents'),
          title='Chart 4: Heatmap, Day of Week vs Hour').show()

### Chart 5: Accidents by month, year wise *(Play)*

In [36]:
d = df.groupby(['Year', 'Month']).size().reset_index(name='Accidents')
px.bar(d, x='Month', y='Accidents', animation_frame='Year', category_orders={'Month': month_order},
       range_y=[0, d['Accidents'].max() * 1.1], title='Chart 5: Accidents by Month').show()

### Chart 6: Accidents by weather, year wise *(Play)*

In [37]:
d = df.groupby(['Year', 'weather']).size().reset_index(name='Accidents')
px.bar(d, x='weather', y='Accidents', color='weather', animation_frame='Year',
       range_y=[0, d['Accidents'].max() * 1.1], title='Chart 6: Accidents by Weather').show()

### Chart 7: Share of severe accidents (major + fatal) by weather
Counts alone mislead (common conditions have more accidents simply because they are common), so we compare the **share** of severe accidents inside each group.

In [38]:
d = sev_wea.reset_index(); d.columns = ['Weather', 'Severe %']
px.bar(d, x='Weather', y='Severe %', color='Weather',
       title='Chart 7: Severe Accident Share by Weather (%)').show()

### Chart 8: Recorded causes, year wise *(Play)*

In [39]:
d = df.groupby(['Year', 'cause']).size().reset_index(name='Accidents')
px.bar(d, x='cause', y='Accidents', color='cause', animation_frame='Year',
       range_y=[0, d['Accidents'].max() * 1.1], title='Chart 8: Recorded Causes of Accidents').show()

### Chart 9: Severe accident share by road feature (%)

In [40]:
d = sev_feat.reset_index(); d.columns = ['Road feature', 'Severe %']
px.bar(d, x='Severe %', y='Road feature', orientation='h',
       title='Chart 9: Severe Accident Share by Road Feature (%)', height=520).show()

### Chart 10: Top 10 states, year wise *(Play)*

In [41]:
d = df[df['state'].isin(states.index)].groupby(['Year', 'state']).size().reset_index(name='Accidents')
px.bar(d, x='state', y='Accidents', color='state', animation_frame='Year',
       category_orders={'state': list(states.index)}, range_y=[0, d['Accidents'].max() * 1.1],
       title='Chart 10: Top 10 States by Recorded Accidents').show()

### Bonus: Accident locations on the map of India, year wise *(Play)*

In [42]:
g = df[df['geo_ok']].sample(min(3000, int(df['geo_ok'].sum())), random_state=1).sort_values('Year')
fig = px.scatter_geo(g, lat='latitude', lon='longitude', color='accident_severity', animation_frame='Year',
                     category_orders={'accident_severity': sev_order}, hover_name='city', opacity=0.6,
                     title='Bonus: Accident Locations (sample of 3,000)')
fig.update_geos(fitbounds='locations', visible=True)
fig.show()

## Step 8: Data-quality gaps

In [43]:
print('Rows in file                     :', n_raw)
print('Duplicate ids                    :', dup_ids)
print('Invalid dates / hours            :', bad_date, '/', bad_hour)
print('Locations outside India bounds   :', bad_geo)
print()
print('Columns with missing values (%):')
print(miss[miss > 0].sort_values(ascending=False))
print()
print('Accidents per hour (min / max)   :', int(hourly.min()), '/', int(hourly.max()))
print('Years in data                    :', sorted(df['Year'].unique()))

Rows in file                     : 20000
Duplicate ids                    : 0
Invalid dates / hours            : 0 / 0
Locations outside India bounds   : 0

Columns with missing values (%):
festival    99.42
dtype: float64

Accidents per hour (min / max)   : 786 / 888
Years in data                    : ['2022', '2023', '2024', '2025']


## Step 9: Key insights (generated from the data)

In [44]:
from IPython.display import Markdown, display

low_hour = int(hourly.idxmin())
weekend_pct = round(float(df['Weekday'].isin(['Saturday', 'Sunday']).mean() * 100), 1)
top_cause_pct = round(float(cause.iloc[0] / len(df) * 100), 1)
insights = [
    f'Accidents peak at {peak_hour}:00 and are lowest at {low_hour}:00. The busiest time period is {period.idxmax()}.',
    f'{wd.idxmax()} has the most accidents ({int(wd.max()):,}) and {wd.idxmin()} the fewest ({int(wd.min()):,}); weekends hold {weekend_pct}% of accidents.',
    f'{severe_pct}% of accidents are major or fatal; the most common severity is {sev.idxmax()}.',
    f'{sev_wea.index[0]} weather has the highest share of severe accidents ({sev_wea.iloc[0]:.1f}%). This is an association, not a cause.',
    f'The most common recorded cause is {cause.index[0]} ({top_cause_pct}% of accidents).',
    f'{states.index[0]} has the most records, and {cities.index[0]} is the top city.',
    f'Among road features, "{sev_feat.index[0]}" has the highest severe share ({sev_feat.iloc[0]:.1f}%).']
display(Markdown('\n'.join(f'{i}. {t}' for i, t in enumerate(insights, 1))))

1. Accidents peak at 2:00 and are lowest at 7:00. The busiest time period is Night (0-5).
2. Monday has the most accidents (2,966) and Wednesday the fewest (2,726); weekends hold 28.6% of accidents.
3. 44.9% of accidents are major or fatal; the most common severity is minor.
4. clear weather has the highest share of severe accidents (45.0%). This is an association, not a cause.
5. The most common recorded cause is distraction (20.1% of accidents).
6. Maharashtra has the most records, and Chandigarh is the top city.
7. Among road features, "traffic_density: medium" has the highest severe share (45.4%).

## Limitations
- The dataset is a Kaggle compilation of 20,000 records (2023-2025). Its collection method should be checked on the Kaggle page; if it is simulated or compiled rather than official police data, results describe this dataset only, not national rates.
- Counts are not adjusted for traffic volume or population, so busy places and times naturally show more accidents.
- Some columns (such as `festival`) are mostly empty.
- Findings show associations, not causes. No blame is assigned.

In [45]:
conclusion = (f'This analysis of {len(df):,} recorded accidents in India shows clear time and place patterns. '
              f'Accidents peak at {peak_hour}:00, with the busiest period being {period.idxmax()}, and {wd.idxmax()} has the most accidents. '
              f'{severe_pct}% of accidents are major or fatal, and the most common recorded cause is {cause.index[0]}. '
              f'{states.index[0]} has the most records. These are patterns only and do not prove causes. '
              'Questions for further study: why do the peak hours stand out, do high-severity road features need better safety measures, '
              'and do the same patterns hold in official, full-year data?')
display(Markdown('## Conclusion\n' + conclusion))

## Conclusion
This analysis of 20,000 recorded accidents in India shows clear time and place patterns. Accidents peak at 2:00, with the busiest period being Night (0-5), and Monday has the most accidents. 44.9% of accidents are major or fatal, and the most common recorded cause is distraction. Maharashtra has the most records. These are patterns only and do not prove causes. Questions for further study: why do the peak hours stand out, do high-severity road features need better safety measures, and do the same patterns hold in official, full-year data?

## Step 10: Download a clear PDF report
Animated charts cannot be saved in a PDF, so this cell builds a static PDF with the main charts and insights.

In [46]:
import textwrap
from matplotlib.backends.backend_pdf import PdfPages
from google.colab import files

def text_page(pdf, title, lines):
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.text(0.08, 0.93, title, fontsize=18, weight='bold')
    y = 0.87
    for ln in lines:
        t = textwrap.fill(ln, 80)
        fig.text(0.08, y, t, fontsize=11, va='top')
        y -= 0.03 * (t.count('\n') + 1) + 0.02
    pdf.savefig(fig); plt.close(fig)

def chart_page(pdf, title, draw, note):
    fig, ax = plt.subplots(figsize=(8.27, 6))
    draw(ax); ax.set_title(title)
    fig.text(0.08, 0.02, textwrap.fill('Insight: ' + note, 90), fontsize=10)
    fig.subplots_adjust(bottom=0.25)
    pdf.savefig(fig); plt.close(fig)

def bars(s, color, horiz=False):
    def f(ax):
        if horiz: s.sort_values().plot.barh(ax=ax, color=color)
        else:
            s.plot.bar(ax=ax, color=color); ax.tick_params(axis='x', rotation=40)
    return f

with PdfPages('Road_Accident_India_Report.pdf') as pdf:
    text_page(pdf, 'Road Accident Pattern Analysis, India (DA-022)', [
        'Domain: Road Safety Analytics',
        'Problem: A road-safety team wants to understand when and where recorded accidents in India are more frequent.',
        'Objectives: (1) Summarize accident counts and severity. (2) Compare time periods and conditions. (3) Identify data-quality gaps.',
        f'Dataset: Indian Road Accident Dataset (2022-2025), Kaggle. {len(df):,} records after cleaning (from {n_raw:,}).',
        'Tools: Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly, Google Colab.',
        'Method: loaded data, removed duplicate ids, parsed dates, checked hours and locations, built weekday/month/period features, built charts.',
        'Note: Descriptive analysis. Correlation does not prove causation and no blame is assigned.'])
    text_page(pdf, 'Key Findings', [f'{i}. {t}' for i, t in enumerate(insights, 1)])
    chart_page(pdf, 'Accidents by Severity', bars(sev, '#c0392b'), f'{sev.idxmax()} is the most common severity; {severe_pct}% are major or fatal.')
    def hp(ax):
        ax.plot(hourly.index, hourly.values, marker='o', color='crimson'); ax.set_xticks(range(24)); ax.set_xlabel('Hour')
    chart_page(pdf, 'Accidents by Hour of Day', hp, f'Peak at {peak_hour}:00; lowest at {low_hour}:00.')
    chart_page(pdf, 'Accidents by Day of Week', bars(wd, '#2980b9'), f'{wd.idxmax()} highest, {wd.idxmin()} lowest.')
    chart_page(pdf, 'Heatmap: Weekday vs Hour', lambda ax: sns.heatmap(pivot, cmap='YlOrRd', ax=ax), 'Darker cells mean more accidents.')
    chart_page(pdf, 'Accidents by Month', bars(mo, '#16a085'), f'{mo.idxmax()} has the most accidents in this data.')
    chart_page(pdf, 'Accidents by Weather', bars(wea, '#8e44ad'), f'{wea.idxmax()} has the most accidents because it is the most common condition.')
    chart_page(pdf, 'Severe Accident Share by Weather (%)', bars(sev_wea, '#e67e22'), f'{sev_wea.index[0]} has the highest severe share ({sev_wea.iloc[0]:.1f}%). Association, not cause.')
    chart_page(pdf, 'Recorded Causes', bars(cause, '#2c3e50'), f'{cause.index[0]} is the most common recorded cause.')
    chart_page(pdf, 'Severe Share by Road Feature (%)', bars(sev_feat.head(10), '#d35400', True), f'"{sev_feat.index[0]}" has the highest severe share.')
    chart_page(pdf, 'Top 10 States', bars(states, '#34495e'), f'{states.index[0]} has the most records.')
    text_page(pdf, 'Data Quality, Limitations and Conclusion', [
        f'Rows in file: {n_raw:,}. Duplicate ids: {dup_ids}. Invalid dates: {bad_date}. Invalid hours: {bad_hour}. Locations outside India bounds: {bad_geo}.',
        'Columns with missing values: ' + ', '.join(f'{c} ({v}%)' for c, v in miss[miss > 0].sort_values(ascending=False).items()) if (miss > 0).any() else 'No columns with missing values.',
        'Limitations: the collection method of the Kaggle dataset should be verified; counts are not adjusted for traffic volume; some columns are mostly empty.',
        'Findings show associations, not causes.',
        'Conclusion: ' + conclusion])
files.download('Road_Accident_India_Report.pdf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 11: Download the dashboard (single HTML page)
Creates `Road_Accident_India_Dashboard.html`. Open it in Chrome (internet needed for the charts library).

In [47]:
from google.colab import files

def sdf(s):
    d = s.reset_index(); d.columns = ['k', 'v']; d['k'] = d['k'].astype(str); return d

def style(fig, h=320, legend=False):
    fig.update_layout(template='plotly_dark', paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                      height=h, margin=dict(l=10, r=10, t=10, b=10), showlegend=legend)
    return fig

def card(title, tag, fig, cls=''):
    return (f'<div class="card {cls}"><div class="ct"><b>{title}</b><span>{tag}</span></div>'
            + fig.to_html(full_html=False, include_plotlyjs=False, config={'responsive': True}) + '</div>')

def b(s, color, h=320, horiz=False):
    d = sdf(s)
    f = px.bar(d, x='v', y='k', orientation='h', color_discrete_sequence=[color]) if horiz else px.bar(d, x='k', y='v', color_discrete_sequence=[color])
    f.update_layout(xaxis_title=None, yaxis_title=None)
    return style(f, h)

f1 = style(px.line(sdf(hourly).assign(k=lambda x: x['k'].astype(int)), x='k', y='v', markers=True, color_discrete_sequence=['#2de0a5']))
f2 = b(sev, '#38b6ff')
f3 = b(wd, '#2de0a5')
f4 = b(sev_wea.head(8), '#f5c518')
f5 = b(cause, '#a78bfa')
f6 = b(sev_feat.head(10).sort_values(), '#fb7185', 340, True)
f7 = b(states, '#38b6ff')
f8 = b(mo, '#2de0a5')
f9 = b(wea, '#f5c518')
f10 = style(px.imshow(pivot, color_continuous_scale='YlOrRd', aspect='auto'), 340)
gm = df[df['geo_ok']].sample(min(2500, int(df['geo_ok'].sum())), random_state=1)
f11 = px.scatter_geo(gm, lat='latitude', lon='longitude', color='accident_severity', hover_name='city', opacity=0.6)
f11.update_geos(fitbounds='locations', visible=True, bgcolor='rgba(0,0,0,0)')
f11 = style(f11, 420, True)

kpis = [('Total Accidents', f'{len(df):,}'), ('Peak Hour', f'{peak_hour}:00'), ('Severe (Major+Fatal)', f'{severe_pct}%'),
        ('Top State', states.index[0]), ('Total Casualties', f'{total_cas:,}'), ('Top Cause', str(cause.index[0]).title())]
kpi_html = ''.join(f'<div class="kpi"><small>{k}</small><div>{v}</div></div>' for k, v in kpis)
ins_html = ''.join(f'<li>{i}</li>' for i in insights)
gap_items = [(c, f'{v}% missing') for c, v in miss[miss > 0].sort_values(ascending=False).head(5).items()]
gap_items += [('Duplicate ids', str(dup_ids)), ('Outside India bounds', str(bad_geo))]
gap_html = ''.join(f'<tr><td>{c}</td><td>{v}</td></tr>' for c, v in gap_items)

css = '''<style>body{margin:0;background:#0b1437;color:#e5e9f5;font-family:Segoe UI,Arial,sans-serif}
.wrap{max-width:1200px;margin:auto;padding:20px}h1{margin:0;font-size:24px}.sub{color:#8fa0c8;margin:4px 0 18px}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:12px;margin-bottom:16px}
.kpi{background:#131f4d;border:1px solid #223070;border-radius:12px;padding:14px}.kpi small{color:#8fa0c8}
.kpi div{font-size:24px;font-weight:700;margin-top:4px;color:#2de0a5}
.grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:14px}.grid>*{min-width:0}
.card{background:#131f4d;border:1px solid #223070;border-radius:12px;padding:12px;overflow:hidden}.full{grid-column:1/-1}
.js-plotly-plot,.plotly-graph-div{max-width:100%!important}
.ct{display:flex;justify-content:space-between;margin-bottom:6px}
.ct span{background:#1e2e6e;border-radius:6px;padding:2px 8px;font-size:12px;color:#8fa0c8}
li{margin:6px 0}table{width:100%;border-collapse:collapse}td{padding:6px;border-bottom:1px solid #223070}
.foot{color:#8fa0c8;font-size:12px;margin-top:16px}@media(max-width:800px){.grid{grid-template-columns:1fr}}</style>'''

html = ('<html><head><meta charset="utf-8"><title>DA-022 India Dashboard</title>'
        '<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>' + css + '</head><body><div class="wrap">'
        '<h1>Road Accident Pattern Analysis, India</h1>'
        '<div class="sub">DA-022 | Road Safety Analytics | Indian Road Accident Dataset (Kaggle)</div>'
        f'<div class="kpis">{kpi_html}</div><div class="grid">'
        + card('Accidents by Hour of Day', 'Chart #1', f1) + card('Severity Distribution', 'Chart #2', f2)
        + card('Accidents by Weekday', 'Chart #3', f3) + card('Severe Share by Weather (%)', 'Chart #4', f4)
        + card('Recorded Causes', 'Chart #5', f5) + card('Severe Share by Road Feature (%)', 'Chart #6', f6)
        + card('Top 10 States', 'Chart #7', f7) + card('Accidents by Month', 'Chart #8', f8)
        + card('Accidents by Weather', 'Chart #9', f9) + card('Heatmap: Weekday vs Hour', 'Chart #10', f10)
        + card('Accident Locations (sample)', 'Map', f11, 'full')
        + f'<div class="card"><div class="ct"><b>Key Insights</b></div><ul>{ins_html}</ul></div>'
        + f'<div class="card"><div class="ct"><b>Data-Quality Gaps</b></div><table>{gap_html}</table></div>'
        + '</div><div class="foot">Descriptive analysis only: patterns, not causes. Counts are not adjusted for traffic volume.</div>'
        '</div></body></html>')

with open('Road_Accident_India_Dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)
files.download('Road_Accident_India_Dashboard.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>